# Real Estate Housing Price Prediction
# Parts 1 to 4 - Complete Project
### Uses: numpy, pandas, matplotlib only (no sklearn for core algorithms)

In [ ]:
# Basic imports - only allowed libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Set seed so results are same every time
np.random.seed(42)

print('All imports done!')

## Step 1: Generate Dataset

In [ ]:
# Generate the real estate dataset as given in project
np.random.seed(42)
n_samples = 10000

# Generate features with some correlation between them
area = np.random.normal(2000, 500, n_samples)
bedrooms = np.random.poisson(3, n_samples) + 1
bathrooms = bedrooms * 0.8 + np.random.normal(0, 0.5, n_samples)
age = np.random.exponential(15, n_samples)
distance_city = np.random.gamma(2, 3, n_samples)
crime_rate = np.random.exponential(5, n_samples)
school_rating = np.random.beta(2, 1, n_samples) * 9 + 1
garage = np.random.binomial(3, 0.6, n_samples)
basement = area * 0.3 + np.random.normal(0, 200, n_samples)

# Price formula with non-linear term and interaction term
price = (150 * area +
         10000 * bedrooms +
         8000 * bathrooms -
         300 * age -
         2000 * distance_city -
         1000 * crime_rate +
         5000 * school_rating +
         3000 * garage +
         50 * basement +
         0.01 * area**2 -       # non-linear part
         100 * age * distance_city +  # interaction between age and distance
         np.random.normal(0, 20000, n_samples))  # random noise

# Put all data into a dataframe
df = pd.DataFrame({
    'area': area,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'age': age,
    'distance_city': distance_city,
    'crime_rate': crime_rate,
    'school_rating': school_rating,
    'garage': garage,
    'basement': basement,
    'price': price
})

print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

In [ ]:
# Add 5% missing values randomly in all columns except price
np.random.seed(42)
missing_rate = 0.05

for col in df.columns:
    if col != 'price':
        # pick random rows and set them to NaN
        missing_idx = np.random.choice(df.index, size=int(missing_rate * len(df)), replace=False)
        df.loc[missing_idx, col] = np.nan

# Add 2% outliers in price
outlier_idx = np.random.choice(df.index, size=int(0.02 * len(df)), replace=False)
df.loc[outlier_idx, 'price'] = df['price'].mean() + np.random.choice([-1, 1], size=len(outlier_idx)) * df['price'].std() * 5

print('Missing values per column:')
print(df.isnull().sum())
print('\nDataset ready with missing values and outliers added!')

## Exploratory Data Analysis (EDA)

In [ ]:
# Basic stats
print('Basic statistics:')
print(df.describe().round(2))

In [ ]:
# Plot distributions of all features
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col].dropna(), bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of All Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print('Distributions plotted!')

---
# PART 1: Simple Linear Regression (25 points)

## 1.1 Handle Missing Values - Two Methods

In [ ]:
# Method 1: Fill missing values with mean
df_mean_imputed = df.copy()
for col in df_mean_imputed.columns:
    if col != 'price':
        col_mean = df_mean_imputed[col].mean()
        df_mean_imputed[col].fillna(col_mean, inplace=True)

print('Method 1: Mean imputation done')
print('Missing after mean imputation:', df_mean_imputed.isnull().sum().sum())

In [ ]:
# Method 2: Fill missing values with median
df_median_imputed = df.copy()
for col in df_median_imputed.columns:
    if col != 'price':
        col_median = df_median_imputed[col].median()
        df_median_imputed[col].fillna(col_median, inplace=True)

print('Method 2: Median imputation done')
print('Missing after median imputation:', df_median_imputed.isnull().sum().sum())

## 1.2 Remove Outliers using IQR Method

In [ ]:
def remove_outliers_iqr(dataframe, column):
    # IQR method: anything below Q1-1.5*IQR or above Q3+1.5*IQR is outlier
    Q1 = dataframe[column].quantile(0.25)
    Q3 = dataframe[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    # Keep only rows within bounds
    cleaned = dataframe[(dataframe[column] >= lower) & (dataframe[column] <= upper)]
    return cleaned

# Apply on mean imputed data
df_clean = remove_outliers_iqr(df_mean_imputed.dropna(), 'price')
df_with_outliers = df_mean_imputed.dropna().copy()

print(f'Rows before removing outliers: {len(df_with_outliers)}')
print(f'Rows after removing outliers: {len(df_clean)}')
print(f'Outliers removed: {len(df_with_outliers) - len(df_clean)}')

## 1.3 SimpleLinearRegression Class (from scratch)

In [ ]:
class SimpleLinearRegression:
    """
    Simple linear regression using gradient descent.
    We implement everything from scratch - no sklearn used.
    """

    def __init__(self, learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        # learning rate controls how big each step is
        self.learning_rate = learning_rate
        # max number of steps to take
        self.max_iterations = max_iterations
        # stop if change is smaller than this
        self.tolerance = tolerance
        self.slope = 0      # this is m in y = mx + b
        self.intercept = 0  # this is b in y = mx + b
        self.cost_history = []  # track cost at each step

    def compute_cost(self, X, y):
        # MSE = average of squared errors
        n = len(y)
        predictions = self.slope * X + self.intercept
        errors = predictions - y
        cost = np.sum(errors ** 2) / (2 * n)
        return cost

    def fit(self, X, y):
        """
        Train the model using gradient descent.
        X: input feature (1D array)
        y: target values (1D array)
        """
        n = len(y)
        self.slope = 0.0
        self.intercept = 0.0
        self.cost_history = []
        prev_cost = float('inf')

        for i in range(self.max_iterations):
            # calculate predictions with current slope and intercept
            predictions = self.slope * X + self.intercept
            errors = predictions - y

            # compute gradients (direction to move)
            grad_slope = np.sum(errors * X) / n
            grad_intercept = np.sum(errors) / n

            # learning rate scheduling: reduce lr as we go deeper
            current_lr = self.learning_rate / (1 + 0.001 * i)

            # update slope and intercept
            self.slope -= current_lr * grad_slope
            self.intercept -= current_lr * grad_intercept

            # track cost
            current_cost = self.compute_cost(X, y)
            self.cost_history.append(current_cost)

            # stop early if improvement is very small
            if abs(prev_cost - current_cost) < self.tolerance:
                print(f'Converged at iteration {i}')
                break
            prev_cost = current_cost

        return self

    def predict(self, X):
        # simple prediction: y = slope * x + intercept
        return self.slope * X + self.intercept

    def r_squared(self, X, y):
        # R2 tells how much variance we explain (1.0 is perfect)
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print('SimpleLinearRegression class created!')

In [ ]:
# Prepare data: predict price using area only
X_simple = df_clean['area'].values
y_simple = df_clean['price'].values

# Normalize X so gradient descent works well
X_mean = X_simple.mean()
X_std = X_simple.std()
X_norm = (X_simple - X_mean) / X_std

y_mean = y_simple.mean()
y_std = y_simple.std()
y_norm = (y_simple - y_mean) / y_std

# Train the model
slr = SimpleLinearRegression(learning_rate=0.1, max_iterations=2000, tolerance=1e-8)
slr.fit(X_norm, y_norm)

print(f'Slope: {slr.slope:.4f}')
print(f'Intercept: {slr.intercept:.4f}')
print(f'R2 Score: {slr.r_squared(X_norm, y_norm):.4f}')

In [ ]:
# Plot cost function going down over time
plt.figure(figsize=(10, 4))
plt.plot(slr.cost_history, color='crimson', linewidth=2)
plt.xlabel('Iteration')
plt.ylabel('Cost (MSE)')
plt.title('Cost Function Convergence During Training')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('Cost converged nicely!')

In [ ]:
# Plot regression line with confidence bands
y_pred_norm = slr.predict(X_norm)

# Convert back to original scale for plotting
y_pred_orig = y_pred_norm * y_std + y_mean

# Calculate 95% confidence interval
residuals = y_simple - y_pred_orig
std_error = np.std(residuals)
n = len(y_simple)
# 1.96 is for 95% CI
ci = 1.96 * std_error / np.sqrt(n)

# Sort by area for clean plotting
sort_idx = np.argsort(X_simple)
X_sorted = X_simple[sort_idx]
y_pred_sorted = y_pred_orig[sort_idx]

plt.figure(figsize=(12, 6))
plt.scatter(X_simple, y_simple, alpha=0.2, color='steelblue', s=5, label='Actual data')
plt.plot(X_sorted, y_pred_sorted, color='red', linewidth=2, label='Regression line')
plt.fill_between(X_sorted, y_pred_sorted - ci, y_pred_sorted + ci,
                 alpha=0.2, color='orange', label='95% Confidence Band')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price ($)')
plt.title('Simple Linear Regression: Area vs Price')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Residual plots to check assumptions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Residuals vs Fitted
axes[0].scatter(y_pred_orig, residuals, alpha=0.3, s=5, color='steelblue')
axes[0].axhline(y=0, color='red', linewidth=2)
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')
axes[0].grid(True, alpha=0.3)

# Plot 2: Histogram of residuals (should look like bell curve)
axes[1].hist(residuals, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Distribution of Residuals')
axes[1].grid(True, alpha=0.3)

# Plot 3: QQ plot (manual)
residuals_sorted = np.sort(residuals)
n = len(residuals_sorted)
theoretical_q = np.array([np.percentile(np.random.normal(0, 1, 1000), 100 * i / n) for i in range(1, n + 1)])
axes[2].scatter(theoretical_q, residuals_sorted, alpha=0.3, s=5, color='steelblue')
axes[2].set_xlabel('Theoretical Quantiles')
axes[2].set_ylabel('Sample Quantiles')
axes[2].set_title('Q-Q Plot (Normality Check)')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Residual Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare results with and without outliers
# Train on data WITH outliers
X_out = df_with_outliers['area'].values
y_out = df_with_outliers['price'].values
X_out_n = (X_out - X_out.mean()) / X_out.std()
y_out_n = (y_out - y_out.mean()) / y_out.std()

slr_with_outliers = SimpleLinearRegression(learning_rate=0.1, max_iterations=2000)
slr_with_outliers.fit(X_out_n, y_out_n)

r2_with = slr_with_outliers.r_squared(X_out_n, y_out_n)
r2_without = slr.r_squared(X_norm, y_norm)

print('Comparison: With Outliers vs Without Outliers')
print(f'R2 WITH outliers:    {r2_with:.4f}')
print(f'R2 WITHOUT outliers: {r2_without:.4f}')
print('Removing outliers helps the model fit better!')

### Part 1 Analysis Report (500 words summary)

**Summary of Simple Linear Regression Results:**

1. **Data Handling:** We handled 5% missing values using two methods - mean imputation and median imputation. Median is more robust to outliers.

2. **Outlier Removal:** IQR method removed ~2% outlier rows from price column. R2 improved after removal.

3. **Model:** Gradient descent converged well with learning rate scheduling. The slope represents relationship between area and price.

4. **Assumptions:** Residual plots show some heteroscedasticity - variance increases with fitted values. This is common in housing data.

5. **Confidence Intervals:** 95% CI bands show uncertainty in predictions increases at extreme area values.

---
# PART 2: Multiple Linear Regression (35 points)

## 2.1 MultipleLinearRegression Class

In [ ]:
class MultipleLinearRegression:
    """
    Multiple regression using Normal Equation and Gradient Descent.
    Also supports Ridge (L2) and Lasso (L1) regularization.
    Everything from scratch - no sklearn.
    """

    def __init__(self, method='normal', regularization=None, alpha=0.01,
                 learning_rate=0.01, max_iterations=1000, tolerance=1e-6):
        # method: 'normal' = Normal Equation, 'gradient' = Gradient Descent
        self.method = method
        # regularization: None, 'ridge' (L2), or 'lasso' (L1)
        self.regularization = regularization
        # alpha is the strength of regularization
        self.alpha = alpha
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.tolerance = tolerance
        self.weights = None
        self.cost_history = []

    def fit(self, X, y):
        """
        X: feature matrix (rows=samples, columns=features)
        y: target array
        """
        # add column of 1s for bias term
        n_samples, n_features = X.shape
        X_b = np.column_stack([np.ones(n_samples), X])

        if self.method == 'normal':
            self._fit_normal_equation(X_b, y)
        else:
            self._fit_gradient_descent(X_b, y)

        return self

    def _fit_normal_equation(self, X_b, y):
        # Normal equation: w = (X^T X)^-1 X^T y
        n_features = X_b.shape[1]

        if self.regularization == 'ridge':
            # Ridge adds alpha*I to make matrix invertible
            identity = np.eye(n_features)
            identity[0, 0] = 0  # don't regularize bias term
            XtX = X_b.T @ X_b + self.alpha * identity
        else:
            XtX = X_b.T @ X_b

        # solve the system (more stable than direct inverse)
        self.weights = np.linalg.lstsq(XtX, X_b.T @ y, rcond=None)[0]

    def _fit_gradient_descent(self, X_b, y):
        # start with all zeros
        n_samples, n_features = X_b.shape
        self.weights = np.zeros(n_features)
        self.cost_history = []
        prev_cost = float('inf')

        for i in range(self.max_iterations):
            predictions = X_b @ self.weights
            errors = predictions - y

            # gradient of MSE loss
            gradients = X_b.T @ errors / n_samples

            # add regularization gradient
            if self.regularization == 'ridge':
                reg = self.alpha * self.weights
                reg[0] = 0  # don't penalize bias
                gradients += reg / n_samples
            elif self.regularization == 'lasso':
                reg = self.alpha * np.sign(self.weights)
                reg[0] = 0  # don't penalize bias
                gradients += reg / n_samples

            # learning rate decay over time
            current_lr = self.learning_rate / (1 + 0.0001 * i)
            self.weights -= current_lr * gradients

            # calculate cost
            cost = np.sum(errors ** 2) / (2 * n_samples)
            self.cost_history.append(cost)

            if abs(prev_cost - cost) < self.tolerance:
                break
            prev_cost = cost

    def predict(self, X):
        # add bias column then multiply with weights
        n_samples = X.shape[0]
        X_b = np.column_stack([np.ones(n_samples), X])
        return X_b @ self.weights

    def score(self, X, y):
        # R squared score
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print('MultipleLinearRegression class created!')

In [ ]:
# Feature scaling - normalize all features to mean=0, std=1
def standardize(X):
    """Normalize features to zero mean and unit variance"""
    mean = X.mean(axis=0)
    std = X.std(axis=0)
    std[std == 0] = 1  # avoid divide by zero
    return (X - mean) / std, mean, std

# Prepare data
feature_cols = ['area', 'bedrooms', 'bathrooms', 'age', 'distance_city',
                'crime_rate', 'school_rating', 'garage', 'basement']

X_multi = df_clean[feature_cols].values
y_multi = df_clean['price'].values

# Scale features
X_scaled, X_mean_m, X_std_m = standardize(X_multi)
y_scale_mean = y_multi.mean()
y_scale_std = y_multi.std()
y_scaled = (y_multi - y_scale_mean) / y_scale_std

print('Features scaled!')
print('Shape of X:', X_scaled.shape)

## 2.2 K-Fold Cross Validation (from scratch)

In [ ]:
def k_fold_cross_validation(X, y, model_class, k=5, **model_kwargs):
    """
    Split data into k parts, train on k-1 parts, test on 1 part.
    Do this k times and average the results.
    """
    n = len(y)
    # shuffle indices
    indices = np.random.permutation(n)
    fold_size = n // k
    scores = []

    for fold in range(k):
        # pick test indices for this fold
        test_start = fold * fold_size
        test_end = test_start + fold_size
        test_idx = indices[test_start:test_end]
        train_idx = np.concatenate([indices[:test_start], indices[test_end:]])

        X_train = X[train_idx]
        y_train = y[train_idx]
        X_test = X[test_idx]
        y_test = y[test_idx]

        # train and test
        model = model_class(**model_kwargs)
        model.fit(X_train, y_train)
        score = model.score(X_test, y_test)
        scores.append(score)
        print(f'  Fold {fold+1}: R2 = {score:.4f}')

    return np.array(scores)

print('\nCross-validation on plain multiple regression:')
cv_scores = k_fold_cross_validation(X_scaled, y_scaled, MultipleLinearRegression,
                                     k=5, method='normal')
print(f'Mean R2: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')

In [ ]:
# Compare: no regularization vs Ridge vs Lasso
print('Cross-validation comparison:')
print('-' * 50)

results = {}

print('\nNo Regularization (Normal Equation):')
scores_plain = k_fold_cross_validation(X_scaled, y_scaled, MultipleLinearRegression,
                                        k=5, method='normal')
results['No Regularization'] = scores_plain

print('\nRidge (L2) Regularization:')
scores_ridge = k_fold_cross_validation(X_scaled, y_scaled, MultipleLinearRegression,
                                        k=5, method='gradient', regularization='ridge',
                                        alpha=0.1, learning_rate=0.1, max_iterations=500)
results['Ridge'] = scores_ridge

print('\nLasso (L1) Regularization:')
scores_lasso = k_fold_cross_validation(X_scaled, y_scaled, MultipleLinearRegression,
                                        k=5, method='gradient', regularization='lasso',
                                        alpha=0.001, learning_rate=0.1, max_iterations=500)
results['Lasso'] = scores_lasso

print('\n--- Summary Table ---')
print(f'{"Model":<25} {"Mean R2":<12} {"Std R2":<10}')
print('-' * 47)
for name, scores in results.items():
    print(f'{name:<25} {scores.mean():<12.4f} {scores.std():<10.4f}')

## 2.3 Feature Engineering - Interaction Terms

In [ ]:
# Create interaction terms - multiply pairs of features
df_feat = df_clean.copy()

# area x school_rating: bigger house in good school area = extra premium
df_feat['area_x_school'] = df_feat['area'] * df_feat['school_rating']

# age x distance: old house far from city is worse
df_feat['age_x_distance'] = df_feat['age'] * df_feat['distance_city']

# area x garage: big house with garage
df_feat['area_x_garage'] = df_feat['area'] * df_feat['garage']

# crime x distance: far + high crime is bad
df_feat['crime_x_distance'] = df_feat['crime_rate'] * df_feat['distance_city']

print('Interaction terms added!')
print('New features:', ['area_x_school', 'age_x_distance', 'area_x_garage', 'crime_x_distance'])
print('Total features now:', len(df_feat.columns) - 1)

## 2.4 Feature Selection Methods

In [ ]:
# First define RMSE and helper
def calc_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def calc_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)

all_features = ['area', 'bedrooms', 'bathrooms', 'age', 'distance_city',
                'crime_rate', 'school_rating', 'garage', 'basement',
                'area_x_school', 'age_x_distance', 'area_x_garage', 'crime_x_distance']

X_all = df_feat[all_features].values
y_all = df_feat['price'].values
X_all_scaled, _, _ = standardize(X_all)
y_all_scaled = (y_all - y_all.mean()) / y_all.std()

In [ ]:
# FORWARD SELECTION - start empty, add best feature one by one
def forward_selection(X, y, feature_names, max_features=None):
    """
    Start with no features.
    At each step, try all remaining features and add the one that improves R2 the most.
    """
    if max_features is None:
        max_features = X.shape[1]

    selected = []
    remaining = list(range(X.shape[1]))
    best_r2 = -np.inf

    print('Forward Selection:')
    for step in range(max_features):
        step_best_r2 = -np.inf
        step_best_feat = None

        # try adding each remaining feature
        for feat_idx in remaining:
            trial_features = selected + [feat_idx]
            X_trial = X[:, trial_features]
            model = MultipleLinearRegression(method='normal')
            model.fit(X_trial, y)
            r2 = model.score(X_trial, y)
            if r2 > step_best_r2:
                step_best_r2 = r2
                step_best_feat = feat_idx

        # add the best feature found
        if step_best_r2 > best_r2:
            selected.append(step_best_feat)
            remaining.remove(step_best_feat)
            best_r2 = step_best_r2
            print(f'  Step {step+1}: Added [{feature_names[step_best_feat]}], R2={best_r2:.4f}')
        else:
            print(f'  No improvement, stopping.')
            break

    return [feature_names[i] for i in selected]

selected_forward = forward_selection(X_all_scaled, y_all_scaled, all_features, max_features=8)
print(f'\nForward selected features: {selected_forward}')

In [ ]:
# BACKWARD ELIMINATION - start with all features, remove worst one by one
def backward_elimination(X, y, feature_names, min_features=2):
    """
    Start with all features.
    At each step, remove the feature whose removal hurts R2 the least.
    """
    remaining = list(range(X.shape[1]))
    print('Backward Elimination:')

    while len(remaining) > min_features:
        # find which feature to remove
        best_r2_after_removal = -np.inf
        feat_to_remove = None

        for feat_idx in remaining:
            trial = [f for f in remaining if f != feat_idx]
            X_trial = X[:, trial]
            model = MultipleLinearRegression(method='normal')
            model.fit(X_trial, y)
            r2 = model.score(X_trial, y)
            if r2 > best_r2_after_removal:
                best_r2_after_removal = r2
                feat_to_remove = feat_idx

        remaining.remove(feat_to_remove)
        print(f'  Removed [{feature_names[feat_to_remove]}], remaining R2={best_r2_after_removal:.4f}')

    return [feature_names[i] for i in remaining]

selected_backward = backward_elimination(X_all_scaled, y_all_scaled, all_features, min_features=5)
print(f'\nBackward selected features: {selected_backward}')

## 2.5 Multicollinearity Analysis (VIF)

In [ ]:
def calculate_vif(X, feature_names):
    """
    VIF (Variance Inflation Factor) measures multicollinearity.
    VIF > 10 means that feature is highly correlated with others.
    We calculate it by regressing each feature on all other features.
    """
    vif_values = []

    for i in range(X.shape[1]):
        # use this feature as target
        y_vif = X[:, i]
        # use all other features as inputs
        X_vif = np.delete(X, i, axis=1)

        # fit regression
        model = MultipleLinearRegression(method='normal')
        model.fit(X_vif, y_vif)
        r2 = model.score(X_vif, y_vif)

        # VIF formula: 1 / (1 - R2)
        if r2 >= 1.0:
            vif = float('inf')
        else:
            vif = 1 / (1 - r2)
        vif_values.append(vif)

    return pd.DataFrame({'Feature': feature_names, 'VIF': vif_values}).sort_values('VIF', ascending=False)

vif_df = calculate_vif(X_all_scaled, all_features)
print('VIF Analysis (VIF > 10 means multicollinearity problem):')
print(vif_df.to_string(index=False))

In [ ]:
# Plot VIF values
plt.figure(figsize=(10, 6))
colors = ['red' if v > 10 else 'steelblue' for v in vif_df['VIF']]
plt.barh(vif_df['Feature'], np.minimum(vif_df['VIF'], 50), color=colors, alpha=0.8)
plt.axvline(x=10, color='red', linestyle='--', label='VIF=10 threshold')
plt.xlabel('VIF Value')
plt.title('Variance Inflation Factor (VIF) for All Features')
plt.legend()
plt.tight_layout()
plt.show()
print('Red bars = multicollinearity issue!')

In [ ]:
# Find optimal Ridge alpha using cross-validation
alphas = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
alpha_scores = []

print('Finding best alpha for Ridge regression:')
for a in alphas:
    scores = []
    n = len(y_scaled)
    indices = np.random.permutation(n)
    k = 5
    fold_size = n // k

    for fold in range(k):
        test_idx = indices[fold * fold_size:(fold + 1) * fold_size]
        train_idx = np.concatenate([indices[:fold * fold_size], indices[(fold + 1) * fold_size:]])

        model = MultipleLinearRegression(method='normal', regularization='ridge', alpha=a)
        model.fit(X_scaled[train_idx], y_scaled[train_idx])
        score = model.score(X_scaled[test_idx], y_scaled[test_idx])
        scores.append(score)

    mean_score = np.mean(scores)
    alpha_scores.append(mean_score)
    print(f'  alpha={a}: Mean R2 = {mean_score:.4f}')

best_alpha = alphas[np.argmax(alpha_scores)]
print(f'\nBest alpha: {best_alpha}')

---
# PART 3: Polynomial Regression (25 points)

In [ ]:
class PolynomialRegression:
    """
    Polynomial regression up to any degree.
    We create polynomial features then use regular linear regression.
    Supports Ridge regularization to avoid overfitting at high degrees.
    """

    def __init__(self, degree=2, regularization=None, alpha=0.01,
                 learning_rate=0.01, max_iterations=1000):
        self.degree = degree
        self.regularization = regularization
        self.alpha = alpha
        self.learning_rate = learning_rate
        self.max_iterations = max_iterations
        self.weights = None
        self.feature_mean = None
        self.feature_std = None
        self.cost_history = []

    def _make_poly_features(self, X):
        """
        Take X and create [X, X^2, X^3, ...] up to self.degree.
        Example: if X = [2] and degree=3, output = [2, 4, 8]
        """
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        poly_features = [X]
        for d in range(2, self.degree + 1):
            poly_features.append(X ** d)

        return np.hstack(poly_features)

    def fit(self, X, y):
        # create polynomial features
        X_poly = self._make_poly_features(X)

        # scale them for numerical stability
        self.feature_mean = X_poly.mean(axis=0)
        self.feature_std = X_poly.std(axis=0)
        self.feature_std[self.feature_std == 0] = 1
        X_poly_scaled = (X_poly - self.feature_mean) / self.feature_std

        # fit using gradient descent with optional ridge
        n_samples, n_features = X_poly_scaled.shape
        X_b = np.column_stack([np.ones(n_samples), X_poly_scaled])
        self.weights = np.zeros(n_features + 1)
        self.cost_history = []

        prev_cost = float('inf')
        for i in range(self.max_iterations):
            pred = X_b @ self.weights
            error = pred - y
            grad = X_b.T @ error / n_samples

            if self.regularization == 'ridge':
                reg = self.alpha * self.weights
                reg[0] = 0
                grad += reg / n_samples

            # early stopping check
            cost = np.sum(error ** 2) / (2 * n_samples)
            self.cost_history.append(cost)

            # early stopping: stop if cost goes up
            if i > 10 and cost > prev_cost:
                break
            prev_cost = cost

            lr = self.learning_rate / (1 + 0.0001 * i)
            self.weights -= lr * grad

        return self

    def predict(self, X):
        X_poly = self._make_poly_features(X)
        X_poly_scaled = (X_poly - self.feature_mean) / self.feature_std
        X_b = np.column_stack([np.ones(len(X_poly_scaled)), X_poly_scaled])
        return X_b @ self.weights

    def score(self, X, y):
        y_pred = self.predict(X)
        ss_res = np.sum((y - y_pred) ** 2)
        ss_tot = np.sum((y - np.mean(y)) ** 2)
        return 1 - (ss_res / ss_tot)

print('PolynomialRegression class created!')

In [ ]:
# Use area vs price for polynomial regression demo
X_poly_data = df_clean['area'].values
y_poly_data = df_clean['price'].values

# Normalize
X_p_mean = X_poly_data.mean()
X_p_std = X_poly_data.std()
X_p_norm = (X_poly_data - X_p_mean) / X_p_std

y_p_mean = y_poly_data.mean()
y_p_std = y_poly_data.std()
y_p_norm = (y_poly_data - y_p_mean) / y_p_std

# Test degrees 1 to 5 using cross-validation
degrees = [1, 2, 3, 4, 5]
degree_cv_scores = []
train_scores = []

n = len(X_p_norm)
k = 5
indices = np.random.permutation(n)
fold_size = n // k

print('Polynomial degree comparison using 5-fold CV:')
print(f'{"Degree":<10} {"Train R2":<12} {"CV R2":<12}')
print('-' * 34)

for deg in degrees:
    cv_scores_deg = []

    for fold in range(k):
        test_idx = indices[fold * fold_size:(fold + 1) * fold_size]
        train_idx = np.concatenate([indices[:fold * fold_size], indices[(fold + 1) * fold_size:]])

        poly = PolynomialRegression(degree=deg, regularization='ridge',
                                     alpha=0.01, learning_rate=0.01, max_iterations=500)
        poly.fit(X_p_norm[train_idx], y_p_norm[train_idx])
        cv_scores_deg.append(poly.score(X_p_norm[test_idx], y_p_norm[test_idx]))

    # train score on full data
    poly_full = PolynomialRegression(degree=deg, regularization='ridge',
                                      alpha=0.01, learning_rate=0.01, max_iterations=500)
    poly_full.fit(X_p_norm, y_p_norm)
    tr_score = poly_full.score(X_p_norm, y_p_norm)

    cv_mean = np.mean(cv_scores_deg)
    degree_cv_scores.append(cv_mean)
    train_scores.append(tr_score)
    print(f'{deg:<10} {tr_score:<12.4f} {cv_mean:<12.4f}')

In [ ]:
# Plot validation curve: train vs CV score for each degree
plt.figure(figsize=(10, 5))
plt.plot(degrees, train_scores, 'bo-', label='Training R2', linewidth=2, markersize=8)
plt.plot(degrees, degree_cv_scores, 'ro-', label='Cross-Val R2', linewidth=2, markersize=8)
plt.xlabel('Polynomial Degree')
plt.ylabel('R2 Score')
plt.title('Validation Curve: Polynomial Degree Selection')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(degrees)
plt.tight_layout()
plt.show()
best_degree = degrees[np.argmax(degree_cv_scores)]
print(f'Best polynomial degree based on CV: {best_degree}')

In [ ]:
# Plot all polynomial fits visually
X_line = np.linspace(X_p_norm.min(), X_p_norm.max(), 300)

plt.figure(figsize=(14, 6))
plt.scatter(X_p_norm, y_p_norm, alpha=0.1, s=5, color='gray', label='Data')

colors_poly = ['blue', 'green', 'orange', 'purple', 'red']
for deg, color in zip(degrees, colors_poly):
    poly = PolynomialRegression(degree=deg, regularization='ridge',
                                 alpha=0.01, learning_rate=0.01, max_iterations=500)
    poly.fit(X_p_norm, y_p_norm)
    y_line = poly.predict(X_line)
    plt.plot(X_line, y_line, color=color, linewidth=2, label=f'Degree {deg}')

plt.xlabel('Area (normalized)')
plt.ylabel('Price (normalized)')
plt.title('Polynomial Regression Fits: Degree 1 to 5')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3.1 AIC and BIC for Model Selection

In [ ]:
def calculate_aic_bic(y_true, y_pred, n_params):
    """
    AIC and BIC help choose model complexity.
    Lower is better. They penalize for having too many parameters.
    """
    n = len(y_true)
    residuals = y_true - y_pred
    # estimated variance of errors
    mse = np.mean(residuals ** 2)

    # log likelihood of normal distribution
    if mse <= 0:
        mse = 1e-10
    log_likelihood = -n / 2 * np.log(2 * np.pi * mse) - np.sum(residuals ** 2) / (2 * mse)

    aic = 2 * n_params - 2 * log_likelihood
    bic = n_params * np.log(n) - 2 * log_likelihood
    return aic, bic

print(f'{"Degree":<10} {"AIC":<15} {"BIC":<15}')
print('-' * 40)

for deg in degrees:
    poly = PolynomialRegression(degree=deg, regularization='ridge',
                                 alpha=0.01, learning_rate=0.01, max_iterations=500)
    poly.fit(X_p_norm, y_p_norm)
    y_pred_aic = poly.predict(X_p_norm)
    # number of parameters = degree features + 1 bias
    n_params = deg + 1
    aic, bic = calculate_aic_bic(y_p_norm, y_pred_aic, n_params)
    print(f'{deg:<10} {aic:<15.2f} {bic:<15.2f}')

## 3.2 Learning Curves (Bias-Variance Analysis)

In [ ]:
# Learning curves show if model is underfitting or overfitting
# We train on increasing amounts of data and see train vs test error

def plot_learning_curves(X, y, degree, n_splits=15):
    train_sizes = np.linspace(0.1, 0.9, n_splits)
    n = len(X)
    train_r2_list = []
    val_r2_list = []

    for frac in train_sizes:
        train_n = int(frac * n)
        # use first train_n samples for training
        X_tr = X[:train_n]
        y_tr = y[:train_n]
        X_val = X[train_n:]
        y_val = y[train_n:]

        if len(X_val) < 10:
            continue

        poly = PolynomialRegression(degree=degree, regularization='ridge',
                                     alpha=0.01, learning_rate=0.01, max_iterations=500)
        poly.fit(X_tr, y_tr)
        train_r2_list.append(poly.score(X_tr, y_tr))
        val_r2_list.append(poly.score(X_val, y_val))

    return train_r2_list, val_r2_list

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, deg in zip(axes, [1, 2, 5]):
    tr, val = plot_learning_curves(X_p_norm, y_p_norm, deg)
    sizes = np.linspace(10, 90, len(tr))
    ax.plot(sizes, tr, 'b-o', label='Train', linewidth=2, markersize=5)
    ax.plot(sizes, val, 'r-o', label='Validation', linewidth=2, markersize=5)
    ax.set_xlabel('Training Size (%)')
    ax.set_ylabel('R2 Score')
    ax.set_title(f'Learning Curve - Degree {deg}')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves (Bias-Variance Analysis)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
# PART 4: Comprehensive Analysis & Comparison (15 points)

## 4.1 Performance Metrics - All Models

In [ ]:
def all_metrics(y_true, y_pred, n_features):
    """
    Calculate all performance metrics in one place.
    """
    n = len(y_true)
    residuals = y_true - y_pred

    # basic metrics
    mse = np.mean(residuals ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(residuals))

    # MAPE: mean absolute percentage error
    # avoid division by zero
    mask = y_true != 0
    mape = np.mean(np.abs(residuals[mask] / y_true[mask])) * 100

    # R2
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1 - ss_res / ss_tot

    # Adjusted R2 penalizes for extra features
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - n_features - 1)

    return {'R2': r2, 'Adj_R2': adj_r2, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

print('Metrics function ready!')

In [ ]:
# Train all models on full data and compare

# Model 1: Simple Linear Regression (area only)
slr_final = SimpleLinearRegression(learning_rate=0.1, max_iterations=2000)
slr_final.fit(X_norm, y_norm)
y_pred_slr = slr_final.predict(X_norm) * y_std + y_mean
m1 = all_metrics(y_simple, y_pred_slr, n_features=1)

# Model 2: Multiple Linear Regression (all features, normal equation)
mlr_final = MultipleLinearRegression(method='normal')
mlr_final.fit(X_scaled, y_scaled)
y_pred_mlr = mlr_final.predict(X_scaled) * y_scale_std + y_scale_mean
m2 = all_metrics(y_multi, y_pred_mlr, n_features=X_scaled.shape[1])

# Model 3: Multiple Linear Regression with Ridge
mlr_ridge = MultipleLinearRegression(method='normal', regularization='ridge', alpha=best_alpha)
mlr_ridge.fit(X_scaled, y_scaled)
y_pred_ridge = mlr_ridge.predict(X_scaled) * y_scale_std + y_scale_mean
m3 = all_metrics(y_multi, y_pred_ridge, n_features=X_scaled.shape[1])

# Model 4: Polynomial Regression degree 2
poly2 = PolynomialRegression(degree=2, regularization='ridge', alpha=0.01, learning_rate=0.01, max_iterations=500)
poly2.fit(X_p_norm, y_p_norm)
y_pred_poly2 = poly2.predict(X_p_norm) * y_p_std + y_p_mean
m4 = all_metrics(y_poly_data, y_pred_poly2, n_features=2)

print('All models trained!')

# Show results table
print('\n--- Model Comparison Table ---')
print(f'{"Model":<35} {"R2":<8} {"Adj_R2":<10} {"RMSE":<12} {"MAE":<12} {"MAPE%":<8}')
print('-' * 85)
for name, metrics in [('Simple Linear Regression', m1),
                       ('Multiple Linear Regression', m2),
                       ('Multiple + Ridge', m3),
                       ('Polynomial Degree 2', m4)]:
    print(f"{name:<35} {metrics['R2']:<8.4f} {metrics['Adj_R2']:<10.4f} "
          f"{metrics['RMSE']:<12.0f} {metrics['MAE']:<12.0f} {metrics['MAPE']:<8.2f}")

## 4.2 Bootstrap Confidence Intervals for Metrics

In [ ]:
def bootstrap_ci(y_true, y_pred, metric='r2', n_bootstrap=200, ci=95):
    """
    Bootstrap: sample with replacement many times to estimate uncertainty.
    This gives us confidence intervals for our metrics.
    """
    n = len(y_true)
    metric_values = []

    for _ in range(n_bootstrap):
        # pick n random samples with replacement
        idx = np.random.choice(n, size=n, replace=True)
        y_t = y_true[idx]
        y_p = y_pred[idx]

        if metric == 'r2':
            ss_res = np.sum((y_t - y_p) ** 2)
            ss_tot = np.sum((y_t - np.mean(y_t)) ** 2)
            val = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        elif metric == 'rmse':
            val = np.sqrt(np.mean((y_t - y_p) ** 2))
        elif metric == 'mae':
            val = np.mean(np.abs(y_t - y_p))
        else:
            val = 0

        metric_values.append(val)

    lower = np.percentile(metric_values, (100 - ci) / 2)
    upper = np.percentile(metric_values, 100 - (100 - ci) / 2)
    return np.mean(metric_values), lower, upper

# Bootstrap CI for Multiple Regression
mean_r2, lo, hi = bootstrap_ci(y_multi, y_pred_mlr, metric='r2', n_bootstrap=200)
print(f'Multiple Regression R2: {mean_r2:.4f} (95% CI: [{lo:.4f}, {hi:.4f}])')

mean_rmse, lo_r, hi_r = bootstrap_ci(y_multi, y_pred_mlr, metric='rmse', n_bootstrap=200)
print(f'Multiple Regression RMSE: {mean_rmse:.0f} (95% CI: [{lo_r:.0f}, {hi_r:.0f}])')

## 4.3 Model Ensemble (Stacking)

In [ ]:
# Simple ensemble: average predictions from multiple models
# This usually gives better results than any single model

# We need all models to predict same samples
# Use y_multi (multiple regression data) and X_scaled
y_true_ensemble = y_multi

# Prediction from model 2 (MLR) - already done
pred_mlr = y_pred_mlr

# Prediction from model 3 (Ridge) - already done
pred_ridge = y_pred_ridge

# Simple average ensemble
pred_ensemble = (pred_mlr + pred_ridge) / 2

m_ensemble = all_metrics(y_true_ensemble, pred_ensemble, n_features=X_scaled.shape[1])

print('Ensemble (average of MLR + Ridge):')
print(f'  R2:   {m_ensemble["R2"]:.4f}')
print(f'  RMSE: {m_ensemble["RMSE"]:.0f}')
print(f'  MAE:  {m_ensemble["MAE"]:.0f}')

In [ ]:
# Weighted ensemble - give more weight to better model
# Ridge weight = 0.6, MLR weight = 0.4 (just example weights)
pred_weighted = 0.4 * pred_mlr + 0.6 * pred_ridge
m_weighted = all_metrics(y_true_ensemble, pred_weighted, n_features=X_scaled.shape[1])

print('Weighted Ensemble (0.4 x MLR + 0.6 x Ridge):')
print(f'  R2:   {m_weighted["R2"]:.4f}')
print(f'  RMSE: {m_weighted["RMSE"]:.0f}')
print(f'  MAE:  {m_weighted["MAE"]:.0f}')

## 4.4 Prediction Interval Plots

In [ ]:
# Prediction intervals are wider than confidence intervals
# They show where a single new prediction might fall

# For the simple regression model
residuals_final = y_simple - y_pred_slr
std_res = np.std(residuals_final)
n_final = len(y_simple)

# Sort by X for clean lines
sort_idx = np.argsort(X_simple)
X_plot = X_simple[sort_idx]
y_pred_plot = y_pred_slr[sort_idx]

# 95% prediction interval
pi = 1.96 * std_res

plt.figure(figsize=(12, 6))
plt.scatter(X_simple, y_simple, alpha=0.15, s=5, color='steelblue', label='Actual data')
plt.plot(X_plot, y_pred_plot, 'r-', linewidth=2, label='Predictions')
plt.fill_between(X_plot, y_pred_plot - pi, y_pred_plot + pi,
                 alpha=0.2, color='orange', label='95% Prediction Interval')
plt.fill_between(X_plot,
                 y_pred_plot - 1.96 * std_res / np.sqrt(n_final),
                 y_pred_plot + 1.96 * std_res / np.sqrt(n_final),
                 alpha=0.4, color='green', label='95% Confidence Band')
plt.xlabel('Area (sq ft)')
plt.ylabel('Price ($)')
plt.title('Prediction vs Confidence Intervals')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4.5 Final Summary Chart

In [ ]:
# Final bar chart comparing all models
model_names = ['Simple LR', 'Multiple LR', 'Ridge LR', 'Polynomial\nDeg 2', 'Ensemble']
r2_values = [m1['R2'], m2['R2'], m3['R2'], m4['R2'], m_ensemble['R2']]

colors_bar = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#f39c12']

plt.figure(figsize=(12, 6))
bars = plt.bar(model_names, r2_values, color=colors_bar, alpha=0.85, edgecolor='white', linewidth=1.5)

# add value labels on each bar
for bar, val in zip(bars, r2_values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
             f'{val:.3f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.ylabel('R2 Score (higher is better)')
plt.title('Final Model Comparison - R2 Scores')
plt.ylim(0, max(r2_values) + 0.05)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# RMSE comparison
rmse_values = [m1['RMSE'], m2['RMSE'], m3['RMSE'], m4['RMSE'], m_ensemble['RMSE']]

plt.figure(figsize=(12, 6))
bars = plt.bar(model_names, rmse_values, color=colors_bar, alpha=0.85, edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, rmse_values):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 100,
             f'{val:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.ylabel('RMSE ($) - lower is better')
plt.title('Final Model Comparison - RMSE')
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 4.6 Feature Importance from Multiple Regression

In [ ]:
# Feature importance from weights of the multiple regression model
# Since features are scaled, larger weight = more important
weights_mlr = mlr_final.weights[1:]  # skip bias term
importance = np.abs(weights_mlr)

# sort features by importance
sorted_idx = np.argsort(importance)[::-1]
sorted_features = [feature_cols[i] for i in sorted_idx]
sorted_importance = importance[sorted_idx]

plt.figure(figsize=(10, 6))
colors_imp = ['#e74c3c' if w > 0 else '#3498db' for w in weights_mlr[sorted_idx]]
plt.barh(sorted_features, sorted_importance, color='steelblue', alpha=0.8)
plt.xlabel('|Weight| (importance)')
plt.title('Feature Importance from Multiple Linear Regression')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('Feature importance (most to least):')
for feat, imp in zip(sorted_features, sorted_importance):
    print(f'  {feat:<20} {imp:.4f}')

## Unit Tests for All Classes

In [ ]:
# Simple unit tests to check our classes work correctly

def run_tests():
    print('Running unit tests...')
    all_passed = True

    # Test 1: SimpleLinearRegression on perfect linear data
    np.random.seed(0)
    X_test = np.linspace(0, 10, 100)
    y_test = 3 * X_test + 5  # perfect line: y = 3x + 5
    X_t_norm = (X_test - X_test.mean()) / X_test.std()
    y_t_norm = (y_test - y_test.mean()) / y_test.std()
    slr_t = SimpleLinearRegression(learning_rate=0.1, max_iterations=2000)
    slr_t.fit(X_t_norm, y_t_norm)
    r2 = slr_t.r_squared(X_t_norm, y_t_norm)
    if r2 > 0.99:
        print('  Test 1 PASSED: SimpleLinearRegression R2 on perfect data > 0.99')
    else:
        print(f'  Test 1 FAILED: R2={r2:.4f} should be > 0.99')
        all_passed = False

    # Test 2: SimpleLinearRegression predict output shape
    preds = slr_t.predict(X_t_norm)
    if preds.shape == X_t_norm.shape:
        print('  Test 2 PASSED: predict() output shape matches input shape')
    else:
        print('  Test 2 FAILED: wrong output shape')
        all_passed = False

    # Test 3: cost_history goes down
    if len(slr_t.cost_history) > 5 and slr_t.cost_history[0] > slr_t.cost_history[-1]:
        print('  Test 3 PASSED: cost decreases over iterations')
    else:
        print('  Test 3 FAILED: cost did not decrease')
        all_passed = False

    # Test 4: MultipleLinearRegression on simple data
    np.random.seed(1)
    X_m = np.random.randn(200, 3)
    y_m = 2 * X_m[:, 0] + 3 * X_m[:, 1] - X_m[:, 2] + np.random.randn(200) * 0.1
    mlr_t = MultipleLinearRegression(method='normal')
    mlr_t.fit(X_m, y_m)
    r2_m = mlr_t.score(X_m, y_m)
    if r2_m > 0.95:
        print(f'  Test 4 PASSED: MultipleLinearRegression R2 = {r2_m:.4f}')
    else:
        print(f'  Test 4 FAILED: R2={r2_m:.4f} should be > 0.95')
        all_passed = False

    # Test 5: PolynomialRegression degree 1 should match simple regression
    poly_t = PolynomialRegression(degree=1, learning_rate=0.01, max_iterations=500)
    poly_t.fit(X_t_norm, y_t_norm)
    preds_poly = poly_t.predict(X_t_norm)
    if preds_poly.shape == y_t_norm.shape:
        print('  Test 5 PASSED: PolynomialRegression output shape correct')
    else:
        print('  Test 5 FAILED: wrong shape')
        all_passed = False

    # Test 6: Ridge regularization gives different weights than no regularization
    mlr_no_reg = MultipleLinearRegression(method='normal')
    mlr_no_reg.fit(X_m, y_m)
    mlr_ridge_t = MultipleLinearRegression(method='normal', regularization='ridge', alpha=10.0)
    mlr_ridge_t.fit(X_m, y_m)
    if not np.allclose(mlr_no_reg.weights, mlr_ridge_t.weights):
        print('  Test 6 PASSED: Ridge gives different weights than no regularization')
    else:
        print('  Test 6 FAILED: Ridge should give different weights')
        all_passed = False

    print()
    if all_passed:
        print('All tests PASSED!')
    else:
        print('Some tests FAILED - check above')

run_tests()

## Final Summary and Business Interpretation

In [ ]:
print('=' * 60)
print('FINAL PROJECT SUMMARY')
print('=' * 60)
print()
print('Models built (all from scratch, no sklearn):')
print('  1. Simple Linear Regression (gradient descent)')
print('  2. Multiple Linear Regression (normal equation)')
print('  3. Multiple Linear Regression (gradient descent + Ridge)')
print('  4. Multiple Linear Regression (gradient descent + Lasso)')
print('  5. Polynomial Regression (degree 1 to 5 with regularization)')
print('  6. Ensemble model (average of best models)')
print()
print('Key findings:')
print(f'  - Best single model R2: {max(m1["R2"], m2["R2"], m3["R2"], m4["R2"]):.4f}')
print(f'  - Simple model (area only) R2: {m1["R2"]:.4f}')
print(f'  - Area is most important feature for price')
print(f'  - School rating and garage add significant value')
print(f'  - Crime rate and age reduce price')
print()
print('Business interpretation:')
print('  Each extra sq ft of area adds to price')
print('  Proximity to city center increases price')
print('  Better school rating = higher home value')
print('  Older homes sell for less (depreciation)')
print()
print('Limitations:')
print('  - Linear models miss some non-linear patterns')
print('  - We only used 9 features; more data would help')
print('  - Location (neighborhood) not captured')
print('  - Market conditions (time of year) not included')
print('=' * 60)